# Fitts' Law + Multi-Dwell Replay Analysis

Analyzes head-controlled pointing data with variance-matched filters and reconstructs throughput for multiple simulated dwell times from a single 2-second experiment.

## How to Use
1. Run Cell 1 (Setup)
2. Run Cell 2 (Upload) - upload your 4 exported files:
   - `fitts-experiment-raw-data-*.csv`
   - `fitts-experiment-results-*.csv`
   - `fitts-cursor-paths-*.json`
   - `fitts-variance-measurement-*.csv` (optional)
3. Run all remaining cells (Runtime > Run all)
4. Download plots from the Files panel

In [ ]:
# Cell 1 - Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import math
import glob
import os
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

OUTPUT_DIR = 'plots'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Packages loaded.')

In [ ]:
# Cell 2 - Upload files
from google.colab import files
import zipfile

print('Upload your experiment files (raw CSV, results CSV, cursor-paths JSON, variance CSV):')
print('You can also upload a .zip containing all of them.\n')
uploaded = files.upload()

for fn in list(uploaded.keys()):
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall('.')
        print(f'Extracted {fn}')

# Auto-detect files
raw_files = sorted(glob.glob('**/fitts-experiment-raw-data*.csv', recursive=True))
results_files = sorted(glob.glob('**/fitts-experiment-results*.csv', recursive=True))
paths_files = sorted(glob.glob('**/fitts-cursor-paths*.json', recursive=True))
variance_files = sorted(glob.glob('**/fitts-variance-measurement*.csv', recursive=True))

print(f'\nDetected files:')
print(f'  Raw data CSVs:       {len(raw_files)}')
print(f'  Results CSVs:        {len(results_files)}')
print(f'  Cursor-paths JSONs:  {len(paths_files)}')
print(f'  Variance CSVs:       {len(variance_files)}')

In [ ]:
# Cell 3 - Load data
raw_dfs = []
for i, f in enumerate(raw_files, 1):
    df = pd.read_csv(f)
    df['ParticipantID'] = f'P{i}'
    df['_source_file'] = f
    raw_dfs.append(df)
    print(f'  P{i}: {len(df)} trials from {os.path.basename(f)}')

raw = pd.concat(raw_dfs, ignore_index=True) if raw_dfs else pd.DataFrame()
print(f'\nTotal raw trials: {len(raw)}')

# Load pre-computed results if available
res_dfs = []
for i, f in enumerate(results_files, 1):
    df = pd.read_csv(f)
    df['ParticipantID'] = f'P{i}'
    res_dfs.append(df)
res_precomputed = pd.concat(res_dfs, ignore_index=True) if res_dfs else None

# Load cursor paths
all_cursor_paths = []
for f in paths_files:
    with open(f) as fh:
        data = json.load(fh)
    all_cursor_paths.extend(data)
    print(f'  Cursor paths: {len(data)} trials from {os.path.basename(f)}')

# Load variance measurements
var_dfs = []
for f in variance_files:
    df = pd.read_csv(f)
    var_dfs.append(df)
variance_meas = pd.concat(var_dfs, ignore_index=True) if var_dfs else None

print(f'\nData loaded: {len(raw)} trials, {len(all_cursor_paths)} cursor paths')
display(raw.head(3))

---
## Part A: Standard Fitts' Law Analysis

In [ ]:
# Cell 4 - Compute Fitts metrics per condition
# Uses kinematic MT (MovementTime) which excludes dwell time.
# Also computes EntryBasedMT metrics for comparison.

def compute_fitts_condition(group, mt_col='MovementTime'):
    """Compute Fitts metrics for a group of trials in one condition."""
    valid = group.dropna(subset=[mt_col])
    if len(valid) < 3:
        return None

    # Screen center for correcting target positions
    # (exported TargetX/Y may be relative to startPoint, not screen center)
    if 'Direction' in valid.columns and 'Amplitude' in valid.columns:
        # Derive screen center from the first trial whose TrialInLayout == 0
        first_trials = valid[valid.get('TrialInLayout', valid['DirectionIndex']) == 0]
        if len(first_trials) > 0:
            cx = first_trials.iloc[0]['StartX']
            cy = first_trials.iloc[0]['StartY']
        else:
            cx = valid.iloc[0]['StartX']
            cy = valid.iloc[0]['StartY']

        rad = np.radians(valid['Direction'].astype(float))
        actual_tx = cx + valid['Amplitude'].astype(float) * np.cos(rad)
        actual_ty = cy + valid['Amplitude'].astype(float) * np.sin(rad)
    else:
        actual_tx = valid['TargetX']
        actual_ty = valid['TargetY']

    # Use kinematic endpoint (EndpointX/Y) if available, else SelectionX/Y
    if 'EndpointX' in valid.columns:
        ex = valid['EndpointX'].astype(float)
        ey = valid['EndpointY'].astype(float)
    else:
        ex = valid['SelectionX'].astype(float)
        ey = valid['SelectionY'].astype(float)

    sx = valid['StartX'].astype(float)
    sy = valid['StartY'].astype(float)

    # Effective amplitude
    Ae = np.sqrt((ex - sx)**2 + (ey - sy)**2).mean()

    # Directional projection for We
    dx_end = ex.values - actual_tx.values
    dy_end = ey.values - actual_ty.values
    dx_axis = actual_tx.values - sx.values
    dy_axis = actual_ty.values - sy.values
    axis_len = np.sqrt(dx_axis**2 + dy_axis**2)
    axis_len = np.where(axis_len == 0, 1, axis_len)
    projections = (dx_end * dx_axis + dy_end * dy_axis) / axis_len

    We = 4.133 * np.std(projections, ddof=1) if len(projections) > 1 else 1.0
    We = max(We, 0.01)
    IDe = np.log2(Ae / We + 1)
    mean_mt = valid[mt_col].mean()
    TP = IDe / mean_mt if mean_mt > 0 else 0

    return {
        'N': len(valid),
        'MeanMT': mean_mt,
        'Ae': Ae,
        'We': We,
        'IDe': IDe,
        'TP': TP,
        'MeanReEntries': valid['ReEntryCount'].mean() if 'ReEntryCount' in valid.columns else None,
    }


# Build conditions table
group_cols = []
for c in ['ParticipantID', 'PairNumber', 'PairVariance', 'FilterType', 'TargetSize', 'Amplitude']:
    if c in raw.columns:
        group_cols.append(c)

rows = []
for keys, grp in raw.groupby(group_cols):
    info = dict(zip(group_cols, keys if isinstance(keys, tuple) else [keys]))
    metrics = compute_fitts_condition(grp, 'MovementTime')
    if metrics:
        info.update(metrics)
        # Also compute entry-based MT metrics
        if 'EntryBasedMT' in grp.columns:
            entry_metrics = compute_fitts_condition(grp, 'EntryBasedMT')
            if entry_metrics:
                info['EntryMT'] = entry_metrics['MeanMT']
                info['EntryTP'] = entry_metrics['TP']
        rows.append(info)

conditions = pd.DataFrame(rows)
print(f'Computed {len(conditions)} conditions')
display(conditions.head())

In [ ]:
# Cell 5 - Summary statistics
print('=' * 70)
print('SUMMARY STATISTICS')
print('=' * 70)
print(f'Participants: {conditions["ParticipantID"].nunique() if "ParticipantID" in conditions.columns else 1}')
print(f'Conditions:   {len(conditions)}')
print(f'Total trials: {len(raw)}')
print(f'\nOverall Throughput: {conditions["TP"].mean():.3f} +/- {conditions["TP"].std():.3f} bits/s')
print(f'Overall MT:         {conditions["MeanMT"].mean():.3f} +/- {conditions["MeanMT"].std():.3f} s')

if 'FilterType' in conditions.columns:
    print('\nBy Filter:')
    display(conditions.groupby('FilterType')[['TP', 'MeanMT', 'We', 'IDe']].agg(['mean', 'std']).round(3))

if 'PairVariance' in conditions.columns:
    print('\nBy SD Level:')
    display(conditions.groupby('PairVariance')[['TP', 'MeanMT', 'We', 'IDe']].agg(['mean', 'std']).round(3))

if 'PairVariance' in conditions.columns and 'FilterType' in conditions.columns:
    print('\nBy SD Level x Filter:')
    display(conditions.groupby(['PairVariance', 'FilterType'])[['TP', 'MeanMT']].agg(['mean', 'std']).round(3))

In [ ]:
# Cell 6 - Statistical tests
print('=' * 70)
print('STATISTICAL TESTS')
print('=' * 70)

filters = sorted(conditions['FilterType'].unique()) if 'FilterType' in conditions.columns else []

def run_comparison(data, label=''):
    if len(filters) != 2:
        return
    f1, f2 = filters
    tp1 = data[data['FilterType'] == f1]['TP'].values
    tp2 = data[data['FilterType'] == f2]['TP'].values
    n = min(len(tp1), len(tp2))
    if n < 2:
        return
    tp1, tp2 = tp1[:n], tp2[:n]
    t_stat, p_val = stats.ttest_rel(tp1, tp2)
    diff = tp1 - tp2
    d = np.mean(diff) / np.std(diff, ddof=1) if np.std(diff, ddof=1) > 0 else 0
    sig = 'SIGNIFICANT' if p_val < 0.05 else 'not significant'
    better = f1 if np.mean(tp1) > np.mean(tp2) else f2
    print(f'\n{label}')
    print(f'  {f1}: {np.mean(tp1):.3f} +/- {np.std(tp1):.3f}')
    print(f'  {f2}: {np.mean(tp2):.3f} +/- {np.std(tp2):.3f}')
    print(f'  t({n-1})={t_stat:.3f}, p={p_val:.4f} -> {sig}')
    print(f'  Cohen\'s d = {abs(d):.3f} ({"small" if abs(d)<0.2 else "medium" if abs(d)<0.5 else "large"})')
    if p_val < 0.05:
        pct = abs(np.mean(tp1) - np.mean(tp2)) / min(np.mean(tp1), np.mean(tp2)) * 100
        print(f'  -> {better} is {pct:.1f}% better')

run_comparison(conditions, 'Overall')

if 'PairVariance' in conditions.columns:
    for sd in sorted(conditions['PairVariance'].unique()):
        subset = conditions[conditions['PairVariance'] == sd]
        run_comparison(subset, f'SD = {sd:.1f}')

In [ ]:
# Cell 7 - Plot: Throughput by Filter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=conditions, x='FilterType', y='TP', palette='Set2', ax=axes[0])
sns.stripplot(data=conditions, x='FilterType', y='TP', color='black', alpha=0.3, ax=axes[0])
axes[0].set_title('Throughput by Filter')
axes[0].set_ylabel('Throughput (bits/s)')
axes[0].set_xlabel('')

sns.boxplot(data=conditions, x='FilterType', y='MeanMT', palette='Set2', ax=axes[1])
sns.stripplot(data=conditions, x='FilterType', y='MeanMT', color='black', alpha=0.3, ax=axes[1])
axes[1].set_title('Movement Time by Filter')
axes[1].set_ylabel('Movement Time (s)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/tp_mt_by_filter.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8 - Plot: Throughput by SD level and filter
if 'PairVariance' in conditions.columns:
    sd_labels = {v: f'SD~{v:.1f}' for v in sorted(conditions['PairVariance'].unique())}
    conditions['SD_Label'] = conditions['PairVariance'].map(sd_labels)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(data=conditions, x='SD_Label', y='TP', hue='FilterType', palette='Set2',
                errorbar='sd', ax=axes[0])
    axes[0].set_title('Throughput by SD Level')
    axes[0].set_ylabel('Throughput (bits/s)')
    axes[0].set_xlabel('SD Level')

    sns.barplot(data=conditions, x='SD_Label', y='MeanMT', hue='FilterType', palette='Set2',
                errorbar='sd', ax=axes[1])
    axes[1].set_title('Movement Time by SD Level')
    axes[1].set_ylabel('Movement Time (s)')
    axes[1].set_xlabel('SD Level')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/tp_mt_by_sd.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No PairVariance column - skipping SD-level plot.')

In [ ]:
# Cell 9 - Fitts' Law regression (MT vs IDe)
fig, ax = plt.subplots(figsize=(10, 6))

palette = sns.color_palette('Set2', n_colors=len(conditions['FilterType'].unique()))
for i, ft in enumerate(sorted(conditions['FilterType'].unique())):
    sub = conditions[conditions['FilterType'] == ft]
    ax.scatter(sub['IDe'], sub['MeanMT'], label=ft, alpha=0.6, s=50, color=palette[i])
    # Regression
    if len(sub) > 2:
        z = np.polyfit(sub['IDe'], sub['MeanMT'], 1)
        x_line = np.linspace(sub['IDe'].min(), sub['IDe'].max(), 100)
        ax.plot(x_line, np.polyval(z, x_line), '--', color=palette[i], lw=2,
                label=f'{ft}: MT = {z[1]:.2f} + {z[0]:.2f} x ID (R²={np.corrcoef(sub["IDe"], sub["MeanMT"])[0,1]**2:.3f})')

ax.set_xlabel('Effective Index of Difficulty (bits)')
ax.set_ylabel('Movement Time (s)')
ax.set_title("Fitts' Law: MT = a + b x IDe")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fitts_law_regression.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 10 - Plot: Throughput by target size
if 'TargetSize' in conditions.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    conditions['TargetSize_round'] = conditions['TargetSize'].round(0).astype(int)
    sns.barplot(data=conditions, x='TargetSize_round', y='TP', hue='FilterType',
                palette='Set2', errorbar='sd', ax=ax)
    ax.set_title('Throughput by Target Size')
    ax.set_ylabel('Throughput (bits/s)')
    ax.set_xlabel('Target Size (px)')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/tp_by_target_size.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 11 - Plot: Re-entry counts
if 'ReEntryCount' in raw.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.boxplot(data=raw, x='FilterType', y='ReEntryCount', palette='Set2', ax=axes[0])
    axes[0].set_title('Re-entries by Filter')
    axes[0].set_ylabel('Re-entry Count')

    if 'PairVariance' in raw.columns:
        raw['SD_Label'] = raw['PairVariance'].round(1).astype(str)
        sns.boxplot(data=raw, x='SD_Label', y='ReEntryCount', hue='FilterType',
                    palette='Set2', ax=axes[1])
        axes[1].set_title('Re-entries by SD Level')
        axes[1].set_ylabel('Re-entry Count')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/reentry_counts.png', dpi=300, bbox_inches='tight')
    plt.show()

    print('Mean re-entries by filter:')
    display(raw.groupby('FilterType')['ReEntryCount'].describe().round(2))

In [ ]:
# Cell 12 - Variance measurement analysis
if variance_meas is not None and len(variance_meas) > 0:
    print('Variance Measurement Results')
    print('=' * 70)
    display(variance_meas)

    if 'MeasuredSD' in variance_meas.columns and 'ExpectedSD' in variance_meas.columns:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(variance_meas['ExpectedSD'], variance_meas['MeasuredSD'], s=80, zorder=5)
        lim = max(variance_meas['ExpectedSD'].max(), variance_meas['MeasuredSD'].max()) * 1.1
        ax.plot([0, lim], [0, lim], 'k--', alpha=0.5, label='Perfect match')
        ax.set_xlabel('Expected SD (px)')
        ax.set_ylabel('Measured SD (px)')
        ax.set_title('Expected vs Measured Filter SD')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/variance_measurement.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print('No variance measurement file uploaded - skipping.')

---
## Part B: Multi-Dwell Replay Analysis

Replays cursor paths at multiple simulated dwell times (0.2s - 2.0s) to reconstruct throughput for each, from the single 2-second experiment.

In [ ]:
# Cell 13 - Multi-dwell replay core functions

def point_in_circle(px, py, cx, cy, r):
    return (px - cx)**2 + (py - cy)**2 <= r**2


def fix_target_positions(cursor_paths):
    """Recompute actual target positions from screen center + direction/amplitude.
    The export stores targetX/Y relative to startPoint (previous target),
    but the experiment hit-test always uses screen center."""
    cx = cursor_paths[0]['startX']
    cy = cursor_paths[0]['startY']
    print(f'Screen center: ({cx}, {cy})')
    for t in cursor_paths:
        rad = math.radians(t['direction'])
        t['actualTargetX'] = cx + t['amplitude'] * math.cos(rad)
        t['actualTargetY'] = cy + t['amplitude'] * math.sin(rad)
    return cx, cy


def simulate_dwell(trial, dwell_ms):
    """Simulate continuous dwell selection. Returns result dict or None."""
    path = trial['cursorPath']
    if not path:
        return None
    tx = trial['actualTargetX']
    ty = trial['actualTargetY']
    radius = trial['targetSize'] / 2
    trial_start = path[0]['t']
    enter_time = None

    for s in path:
        x, y, t = s['x'], s['y'], s['t']
        if point_in_circle(x, y, tx, ty, radius):
            if enter_time is None:
                enter_time = t
            if t - enter_time >= dwell_ms:
                mt = (t - trial_start) / 1000.0 - dwell_ms / 1000.0
                return {'success': True, 'mt': max(mt, 0.001),
                        'endpoint_x': x, 'endpoint_y': y}
        else:
            enter_time = None

    return {'success': False, 'mt': None, 'endpoint_x': None, 'endpoint_y': None}


def compute_tp_for_group(sim_results, metas):
    """ISO 9241-9 throughput from a set of simulated trials."""
    ok = [(s, m) for s, m in zip(sim_results, metas) if s['success']]
    if len(ok) < 3:
        return None
    mts, projs, aes = [], [], []
    for s, m in ok:
        sx, sy = m['startX'], m['startY']
        ex, ey = s['endpoint_x'], s['endpoint_y']
        tx, ty = m['targetX'], m['targetY']
        ae = math.sqrt((ex - sx)**2 + (ey - sy)**2)
        aes.append(ae)
        dx_a, dy_a = tx - sx, ty - sy
        alen = math.sqrt(dx_a**2 + dy_a**2)
        if alen == 0:
            continue
        proj = ((ex - tx)*dx_a + (ey - ty)*dy_a) / alen
        projs.append(proj)
        mts.append(s['mt'])
    if len(mts) < 3:
        return None
    we = max(4.133 * np.std(projs, ddof=1), 0.01)
    ae = np.mean(aes)
    ide = math.log2(ae / we + 1)
    mean_mt = np.mean(mts)
    return {'tp': ide / mean_mt if mean_mt > 0 else 0,
            'mean_mt': mean_mt, 'ae': ae, 'we': we, 'ide': ide,
            'n_success': len(mts), 'n_total': len(sim_results),
            'error_rate': 1 - len(mts)/len(sim_results)}

print('Replay functions defined.')

In [ ]:
# Cell 14 - Run multi-dwell replay
DWELL_TIMES_MS = [200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

if not all_cursor_paths:
    print('No cursor-paths JSON uploaded. Skipping dwell replay.')
    dwell_results = []
else:
    screen_cx, screen_cy = fix_target_positions(all_cursor_paths)

    # Build metadata lookup from raw data
    meta_lookup = {}
    for _, row in raw.iterrows():
        gtn = int(row['GlobalTrialNumber'])
        direction = float(row['Direction'])
        amplitude = float(row['Amplitude'])
        rad = math.radians(direction)
        meta_lookup[gtn] = {
            'pairNumber': int(row['PairNumber']),
            'pairVariance': float(row['PairVariance']),
            'filterType': row['FilterType'],
            'targetSize': float(row['TargetSize']),
            'amplitude': amplitude,
            'direction': direction,
            'startX': float(row['StartX']),
            'startY': float(row['StartY']),
            'targetX': screen_cx + amplitude * math.cos(rad),
            'targetY': screen_cy + amplitude * math.sin(rad),
        }

    dwell_results = []
    for dwell_ms in DWELL_TIMES_MS:
        groups = defaultdict(lambda: {'sim': [], 'meta': []})
        for trial in all_cursor_paths:
            gtn = trial['globalTrialNumber']
            if gtn not in meta_lookup:
                continue
            m = meta_lookup[gtn]
            key = (m['pairNumber'], m['filterType'])
            sim = simulate_dwell(trial, dwell_ms)
            if sim:
                groups[key]['sim'].append(sim)
                groups[key]['meta'].append(m)

        for (pn, ft), g in groups.items():
            r = compute_tp_for_group(g['sim'], g['meta'])
            pv = g['meta'][0]['pairVariance']
            if r:
                dwell_results.append({'dwell_ms': dwell_ms, 'dwell_s': dwell_ms/1000,
                                      'pairNumber': pn, 'pairVariance': pv,
                                      'filterType': ft, **r})
            else:
                dwell_results.append({'dwell_ms': dwell_ms, 'dwell_s': dwell_ms/1000,
                                      'pairNumber': pn, 'pairVariance': pv,
                                      'filterType': ft, 'tp': None, 'mean_mt': None,
                                      'n_success': 0, 'n_total': len(g['sim']),
                                      'error_rate': 1.0})

    dwell_df = pd.DataFrame(dwell_results)
    print(f'Replay complete: {len(dwell_df)} condition-dwell combinations')
    display(dwell_df.head(10))

In [ ]:
# Cell 15 - Plot: Dwell Time vs Throughput (side-by-side by filter)
if dwell_results:
    dwell_df = pd.DataFrame(dwell_results)
    valid = dwell_df.dropna(subset=['tp'])

    filter_types = sorted(valid['filterType'].unique())
    filter_labels = {'exponential': 'Exponential Smoothing', 'oneEuro': 'One Euro Filter'}
    pairs = sorted(valid['pairNumber'].unique())
    colors = sns.color_palette('tab10', n_colors=len(pairs))

    pair_labels = {}
    for _, row in valid.iterrows():
        pair_labels[row['pairNumber']] = f'SD ~ {row["pairVariance"]:.1f}'

    fig, axes = plt.subplots(1, len(filter_types), figsize=(7*len(filter_types), 6), sharey=True)
    if len(filter_types) == 1:
        axes = [axes]

    for ax, ft in zip(axes, filter_types):
        for j, pn in enumerate(pairs):
            sub = valid[(valid['pairNumber'] == pn) & (valid['filterType'] == ft)].sort_values('dwell_s')
            if len(sub) == 0:
                continue
            ax.plot(sub['dwell_s'], sub['tp'], 'o-', color=colors[j],
                    label=pair_labels.get(pn, f'Pair {pn}'), lw=2, ms=6)
        ax.set_xlabel('Dwell Time (s)')
        ax.set_title(filter_labels.get(ft, ft), fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 2.2)

    axes[0].set_ylabel('Throughput (bits/s)')
    fig.suptitle('Dwell Time vs Throughput by SD Level', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/dwell_vs_throughput.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No dwell replay data.')

In [ ]:
# Cell 16 - Plot: Dwell vs Throughput combined (both filters)
if dwell_results:
    valid = dwell_df.dropna(subset=['tp'])
    fig, ax = plt.subplots(figsize=(10, 7))

    filter_styles = {'exponential': ('--', 's'), 'oneEuro': ('-', 'o')}
    filter_short = {'exponential': 'Exp', 'oneEuro': '1\u20ac'}

    for j, pn in enumerate(pairs):
        for ft in filter_types:
            sub = valid[(valid['pairNumber'] == pn) & (valid['filterType'] == ft)].sort_values('dwell_s')
            if len(sub) == 0:
                continue
            ls, mk = filter_styles.get(ft, ('-', 'o'))
            lbl = f'{pair_labels.get(pn, f"P{pn}")} ({filter_short.get(ft, ft)})'
            ax.plot(sub['dwell_s'], sub['tp'], linestyle=ls, marker=mk,
                    color=colors[j], label=lbl, lw=2, ms=6)

    ax.set_xlabel('Dwell Time (s)')
    ax.set_ylabel('Throughput (bits/s)')
    ax.set_title('Dwell Time vs Throughput\n(Dashed = Exponential, Solid = One Euro)',
                 fontweight='bold')
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 2.2)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/dwell_vs_throughput_combined.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 17 - Plot: Dwell vs Error Rate
if dwell_results:
    fig, ax = plt.subplots(figsize=(10, 6))

    for j, pn in enumerate(pairs):
        for ft in filter_types:
            sub = dwell_df[(dwell_df['pairNumber'] == pn) & (dwell_df['filterType'] == ft)].sort_values('dwell_s')
            if len(sub) == 0:
                continue
            ls, mk = filter_styles.get(ft, ('-', 'o'))
            lbl = f'{pair_labels.get(pn, f"P{pn}")} ({filter_short.get(ft, ft)})'
            ax.plot(sub['dwell_s'], sub['error_rate']*100, linestyle=ls, marker=mk,
                    color=colors[j], label=lbl, lw=2, ms=6)

    ax.set_xlabel('Dwell Time (s)')
    ax.set_ylabel('Error Rate (%)')
    ax.set_title('Error Rate vs Dwell Time', fontweight='bold')
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 2.2)
    ax.set_ylim(-2, max(dwell_df['error_rate'].max()*100 + 5, 10))
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/dwell_vs_error_rate.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 18 - Plot: Dwell vs Movement Time
if dwell_results:
    valid = dwell_df.dropna(subset=['mean_mt'])
    fig, ax = plt.subplots(figsize=(10, 6))

    for j, pn in enumerate(pairs):
        for ft in filter_types:
            sub = valid[(valid['pairNumber'] == pn) & (valid['filterType'] == ft)].sort_values('dwell_s')
            if len(sub) == 0:
                continue
            ls, mk = filter_styles.get(ft, ('-', 'o'))
            lbl = f'{pair_labels.get(pn, f"P{pn}")} ({filter_short.get(ft, ft)})'
            ax.plot(sub['dwell_s'], sub['mean_mt'], linestyle=ls, marker=mk,
                    color=colors[j], label=lbl, lw=2, ms=6)

    ax.set_xlabel('Dwell Time (s)')
    ax.set_ylabel('Movement Time (s, dwell excluded)')
    ax.set_title('Movement Time vs Dwell Time\n(MT excludes dwell; longer dwell = harder to sustain)',
                 fontweight='bold')
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 2.2)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/dwell_vs_mt.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell 19 - Dwell replay summary table
if dwell_results:
    print('Dwell Replay Results')
    print('=' * 70)

    pivot = dwell_df.pivot_table(
        index=['pairNumber', 'pairVariance', 'filterType'],
        columns='dwell_s',
        values='tp'
    ).round(3)
    print('\nThroughput (bits/s) by condition and dwell time:')
    display(pivot)

    # Show where throughput drops the most
    print('\nThroughput drop from 0.2s to 2.0s dwell:')
    for _, row in dwell_df.groupby(['pairNumber', 'filterType']).apply(
        lambda g: pd.Series({
            'pairVariance': g['pairVariance'].iloc[0],
            'tp_0.2': g[g['dwell_s'] == 0.2]['tp'].values[0] if len(g[g['dwell_s'] == 0.2]) > 0 else None,
            'tp_2.0': g[g['dwell_s'] == 2.0]['tp'].values[0] if len(g[g['dwell_s'] == 2.0]) > 0 else None,
        })
    ).dropna().iterrows():
        pn, ft = _
        drop_pct = (1 - row['tp_2.0'] / row['tp_0.2']) * 100 if row['tp_0.2'] > 0 else 0
        print(f'  Pair {pn} ({ft}) SD~{row["pairVariance"]:.1f}: '
              f'{row["tp_0.2"]:.3f} -> {row["tp_2.0"]:.3f} ({drop_pct:.1f}% drop)')

---
## Part C: Export & Download

In [ ]:
# Cell 20 - Export all CSVs
conditions.to_csv(f'{OUTPUT_DIR}/fitts_conditions.csv', index=False)
print(f'Saved: {OUTPUT_DIR}/fitts_conditions.csv')

if dwell_results:
    dwell_df.to_csv(f'{OUTPUT_DIR}/multi_dwell_results.csv', index=False)
    print(f'Saved: {OUTPUT_DIR}/multi_dwell_results.csv')

# Summary table
if 'PairVariance' in conditions.columns and 'FilterType' in conditions.columns:
    summary = conditions.groupby(['PairVariance', 'FilterType']).agg({
        'TP': ['mean', 'std'],
        'MeanMT': ['mean', 'std'],
        'We': 'mean',
        'IDe': 'mean',
        'N': 'sum'
    }).round(4)
    summary.to_csv(f'{OUTPUT_DIR}/summary_by_sd_filter.csv')
    print(f'Saved: {OUTPUT_DIR}/summary_by_sd_filter.csv')

print('\nAll output files:')
for f in sorted(glob.glob(f'{OUTPUT_DIR}/*')):
    size = os.path.getsize(f)
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Cell 21 - Download all outputs as zip
import shutil
shutil.make_archive('fitts_analysis_output', 'zip', OUTPUT_DIR)
files.download('fitts_analysis_output.zip')
print('Download started: fitts_analysis_output.zip')